In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# **Smart MCQ Solver — Model 1**

## TF-IDF + Logistic Regression
## Our first main model — a classical ML approach (the "additional model of choice" category), unlike the untrained baseline Approach: Combine prompt + all 5 options into one text → convert to TF-IDF vectors (unigrams to trigrams) → train a Logistic Regression classifier to predict the correct option → rank predicted probabilities to get top-3 answers for MAP@3. Why it matters: Shows how far a simple, fast, interpretable classical ML model can go using strong TF-IDF features, without any neural network involved.


## Imports & Configuration

In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import wandb

print("Libraries loaded.")


Libraries loaded.


In [3]:
DATA_PATH   = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OPTION_COLS = ["A", "B", "C", "D", "E"]


## **W&B Setup**  

In [4]:
os.environ["WANDB_SILENT"] = "true"

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

wandb.init(
    project="smart-mcq-solver",
    name="model1-tfidf-logistic",
    config={
        "model"        : "LogisticRegression",
        "max_features" : 100000,
        "ngram_range"  : "(1,3)",
        "C"            : 10.0,
        "max_iter"     : 2000,
        "sublinear_tf" : True
    }
)
print("W&B ready.")


W&B ready.


## Load Dataset 

In [5]:
train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print("Train:", train.shape)
print("Test :", test.shape)
train.head(3)


Train: (2000, 8)
Test : (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


## **Combine Prompt + Options Into Single Text**
## TF-IDF needs a single text field per question, so the prompt and all 5 options are combined into one string — this ensures option-specific words are captured in the vocabulary too. 

In [6]:
def combine_text(row):
    parts = [str(row['prompt'])] + [str(row[c]) for c in OPTION_COLS]
    return ' '.join(parts)

train['combined'] = train.apply(combine_text, axis=1)
test['combined']  = test.apply(combine_text, axis=1)

print("Text combined.")


Text combined.


## **TF-IDF Vectorization**

In [7]:
vectorizer = TfidfVectorizer(
    max_features=100000,
    ngram_range=(1, 3),
    stop_words='english',
    sublinear_tf=True,
    min_df=1,
    analyzer='word'
)

X_train = vectorizer.fit_transform(train['combined'])
X_test  = vectorizer.transform(test['combined'])

le = LabelEncoder()
y_train = le.fit_transform(train['answer'])

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Classes:", le.classes_)


X_train: (2000, 25642)
X_test : (500, 25642)
Classes: ['A' 'B' 'C' 'D' 'E']


## **Train Logistic Regression**

In [8]:
clf = LogisticRegression(
    max_iter=2000,
    C=10.0,
    solver='lbfgs',
    n_jobs=-1
)

clf.fit(X_train, y_train)
print("Model trained.")


Model trained.


##  **Evaluate MAP@3**

In [9]:
def map_at_3_logreg(ground_truth, predictions):
    score = 0.0
    for k, pred in enumerate(predictions[:3], start=1):
        if pred == ground_truth:
            score = 1.0 / k
            break
    return score

probs      = clf.predict_proba(X_train)
top3_idx   = np.argsort(probs, axis=1)[:, ::-1][:, :3]
top3_preds = [[le.classes_[i] for i in row] for row in top3_idx]

scores     = [map_at_3_logreg(true, pred) for true, pred in zip(train['answer'], top3_preds)]
train_map3_logreg = np.mean(scores)
print(f"Train MAP@3: {train_map3_logreg:.4f}")

wandb.log({"train_map3": train_map3_logreg})


Train MAP@3: 1.0000


## **Generate Submission**

In [10]:
probs_test = clf.predict_proba(X_test)
top3_idx   = np.argsort(probs_test, axis=1)[:, ::-1][:, :3]
test_preds = [' '.join([le.classes_[i] for i in row]) for row in top3_idx]

submission = pd.DataFrame({
    'ID'         : test['id'],
    'Prediction' : test_preds
})
submission.to_csv('submission_logreg.csv', index=False)
print("Submission created as submission_logreg.csv")
print(submission.head(10))

wandb.finish()


Submission created as submission_logreg.csv
   ID Prediction
0   1      A B E
1   2      B C A
2   3      B C D
3   4      E C B
4   5      C D E
5   6      D C A
6   7      E C B
7   8      B C D
8   9      C B D
9  10      B C D


---
## **Summary**

| Metric | Value |
|---|---|
| Model | TF-IDF + Logistic Regression |
| Category | Additional model of choice (classical ML) |
| Training required | Yes |
| Kaggle MAP@3 | 0.73815 |

This model clears the competition's qualifying cutoff (0.73) and
significantly outperforms the untrained baseline (0.31), confirming it
learns genuine patterns from the labeled data rather than relying on
simple word-overlap.
